# Reference-based transfer of niche and domain annotations

Fit molecular and spatial classifiers on curated reference cells, evaluate the excluded reference batch, and optionally transfer annotations to a second assay with a shared-gene model.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import anndata as ad
import cellcharter as cc
import h5py
import joblib
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import squidpy as sq
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/01_domain_discovery").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "domain_reference_v2")
OUT = ROOT / "outputs/domain_discovery" / RUN_ID
REF_H5AD = OUT / "03_domains.h5ad"
COSMX_H5AD = ROOT / "data/processed/GSE308624_gastric_cancer_smi_processed.h5ad"
FULL_MODEL_DIR = OUT / "transfer_full"
SHARED_MODEL_DIR = OUT / "transfer_shared"
HELD_OUT_BATCH = "proseg_8_annotated"
NICHE_KEY = "niche"
DOMAIN_KEY = "domain"
BATCH_KEY = "batch"
REFERENCE_CORE_KEY = "core_id"
TARGET_SAMPLE_KEY = "sample"
N_COMPONENTS = 50
NICHE_HOPS = 3
DOMAIN_HOPS = 7
MAX_PCA_CELLS = 200000
MAX_CELLS_PER_CLASS = 30000
CONFIDENCE_THRESHOLD = 0.5
READ_BATCH_SIZE = 200000
MLP_MAX_ITER = 200
RUN_CROSS_PLATFORM = True
SEED = 0
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc._io import require_new

Read the reference metadata and gene dictionary. Count matrices are streamed later, rather than loading the full reference expression matrix into memory.

In [ ]:
def read_annotation(f, col):
    """Decode categorical annotations without reading the full expression matrix."""
    o = f["obs"][col]
    if isinstance(o, h5py.Group) and "categories" in o:
        cats = [c.decode() if isinstance(c, bytes) else c for c in o["categories"][:]]
        return pd.Categorical.from_codes(o["codes"][:], categories=cats)
    a = o[:]
    return a.astype(str) if a.dtype.kind == "S" else a

In [ ]:
def read_metadata(path, cols):
    """Read the ordered gene dictionary, selected metadata and physical coordinates."""
    with h5py.File(path, "r") as f:
        genes = [
            g.decode() if isinstance(g, bytes) else g for g in f["var"]["_index"][:]
        ]
        obs = pd.DataFrame({c: read_annotation(f, c) for c in cols})
        spatial = f["obsm"]["spatial"][:]
    return (obs, spatial, genes)

In [ ]:
if not REF_H5AD.is_file():
    raise FileNotFoundError(f"Run domain discovery first: {REF_H5AD}")
(reference_obs, reference_xy, reference_genes) = read_metadata(
    REF_H5AD, [BATCH_KEY, REFERENCE_CORE_KEY, NICHE_KEY, DOMAIN_KEY]
)
print(
    {
        "reference_cells": len(reference_obs),
        "reference_genes": len(reference_genes),
        "held_out_batch": HELD_OUT_BATCH,
    }
)

## 2. Fit the reference PCA and annotation classifiers

Exclude the designated batch before PCA and classifier fitting. Normalize counts after gene selection; train the niche classifier on aggregated PCA features and the domain classifier on aggregated reference niche labels.

In [ ]:
def iter_count_batches(path, layer="layers/counts", b=READ_BATCH_SIZE):
    """Stream CSR counts in bounded row chunks; model fitting never implicitly uses normalized X."""
    with h5py.File(path, "r") as f:
        g = f[layer]
        shape = tuple(g.attrs["shape"])
        ip = g["indptr"][:]
        n = shape[0]
        for s in range(0, n, b):
            e = min(s + b, n)
            (p0, p1) = (int(ip[s]), int(ip[e]))
            yield (
                s,
                e,
                sp.csr_matrix(
                    (
                        g["data"][p0:p1].astype(np.float32),
                        g["indices"][p0:p1],
                        ip[s : e + 1] - p0,
                    ),
                    shape=(e - s, shape[1]),
                ),
            )

In [ ]:
def log_normalize_counts(c, t=10000.0):
    """Normalize after gene restriction and apply log1p; empty cells remain zero."""
    c = c.astype(np.float32).copy()
    rs = np.asarray(c.sum(1)).ravel()
    rs[rs == 0] = 1
    c = c.multiply((t / rs)[:, None]).tocsr()
    c.data = np.log1p(c.data)
    return c

In [ ]:
def project_counts(c, mean, comp):
    """Apply the reference-fitted PCA centering and components without refitting targets."""
    Vt = comp.T
    return (np.asarray(c.dot(Vt)) - (mean @ Vt)[None, :]).astype(np.float32)

In [ ]:
def save_pca(p, mean, comp, genes):
    """Persist PCA parameters together with the exact gene order."""
    np.savez(p, mean=mean, components=comp, genes=np.array(genes, dtype=object))

In [ ]:
def one_hot_annotations(s, cats):
    """Encode a fixed reference niche vocabulary; unknown labels contribute zero vectors."""
    return (
        pd.get_dummies(pd.Categorical(s, categories=cats))
        .reindex(columns=cats, fill_value=0)
        .values.astype(np.float32)
    )

In [ ]:
def sample_classes(y, m):
    """Cap reference cells per class before a stratified fitting/hold-out split."""
    idx = []
    for c in pd.unique(y):
        p = np.where(y == c)[0]
        if p.size > m:
            p = rng.choice(p, m, replace=False)
        idx.append(p)
    idx = np.concatenate(idx)
    rng.shuffle(idx)
    return idx

In [ ]:
def train_classifier(X, y, name):
    """Fit standardized MLPs with internal early stopping. Random-cell hold-out is a fitting diagnostic, not independent specimen validation."""
    from sklearn.metrics import classification_report
    from sklearn.model_selection import train_test_split

    le = LabelEncoder()
    ye = le.fit_transform(y)
    sub = sample_classes(ye, MAX_CELLS_PER_CLASS)
    (Xtr, Xte, ytr, yte) = train_test_split(
        X[sub], ye[sub], test_size=0.1, stratify=ye[sub], random_state=SEED
    )
    clf = Pipeline(
        [
            ("s", StandardScaler()),
            (
                "m",
                MLPClassifier(
                    hidden_layer_sizes=(256, 128),
                    alpha=0.0001,
                    batch_size=512,
                    max_iter=MLP_MAX_ITER,
                    early_stopping=True,
                    n_iter_no_change=10,
                    random_state=SEED,
                ),
            ),
        ]
    )
    clf.fit(Xtr, ytr)
    print(f"=== {name} 内部留出 ===")
    print(
        classification_report(
            le.inverse_transform(yte), le.inverse_transform(clf.predict(Xte))
        )
    )
    return {"pipeline": clf, "classes": le.classes_}

In [ ]:
def fit_transfer_bundle(out_dir, gene_subset=None):
    """Fit reference PCA and sequential niche/domain classifiers while excluding the designated batch. Domain fitting uses reference niche labels."""
    (obs, spatial, genes) = read_metadata(
        REF_H5AD, [REFERENCE_CORE_KEY, BATCH_KEY, NICHE_KEY, DOMAIN_KEY]
    )
    cats = list(pd.Categorical(obs[NICHE_KEY]).categories)
    json.dump(
        cats,
        open(os.path.join(out_dir, "niche_categories.json"), "w"),
        ensure_ascii=False,
        indent=2,
    )
    col = (
        np.arange(len(genes))
        if gene_subset is None
        else np.array([{g: i for (i, g) in enumerate(genes)}[g] for g in gene_subset])
    )
    used = genes if gene_subset is None else list(gene_subset)
    tmask = (
        (obs[BATCH_KEY].astype(str) != HELD_OUT_BATCH).values
        & ~pd.isna(obs[NICHE_KEY]).values
        & ~pd.isna(obs[DOMAIN_KEY]).values
    )
    tpos = np.where(tmask)[0]
    print(f"[{out_dir}] 基因{len(used)} 训练细胞{tpos.size}")
    fc = np.sort(rng.choice(tpos, min(MAX_PCA_CELLS, tpos.size), replace=False))
    fs = set(fc.tolist())
    bl = []
    for s, e, sub in iter_count_batches(REF_H5AD):
        loc = [i - s for i in range(s, e) if i in fs]
        if loc:
            bl.append(sub[loc][:, col])
        if e >= fc.max() + 1:
            break
    p = PCA(n_components=N_COMPONENTS, svd_solver="randomized", random_state=SEED)
    p.fit(np.asarray(log_normalize_counts(sp.vstack(bl).tocsr()).todense()))
    (mean, comp) = (p.mean_.astype(np.float32), p.components_.astype(np.float32))
    save_pca(os.path.join(out_dir, "pca.npz"), mean, comp, used)
    del bl
    scores = np.empty((obs.shape[0], N_COMPONENTS), np.float32)
    for s, e, sub in iter_count_batches(REF_H5AD):
        scores[s:e] = project_counts(log_normalize_counts(sub[:, col]), mean, comp)
    a = ad.AnnData(X=scores[tpos], obs=obs.iloc[tpos].reset_index(drop=True))
    a.obsm["X_pca_rep"] = scores[tpos]
    a.obsm["spatial"] = spatial[tpos]
    a.obs[REFERENCE_CORE_KEY] = (
        a.obs[REFERENCE_CORE_KEY].astype("category").cat.remove_unused_categories()
    )
    sq.gr.spatial_neighbors(
        a,
        library_key=REFERENCE_CORE_KEY,
        coord_type="generic",
        delaunay=True,
        spatial_key="spatial",
        percentile=99,
    )
    cc.gr.aggregate_neighbors(
        a,
        n_layers=NICHE_HOPS,
        use_rep="X_pca_rep",
        out_key="X_emb_niche",
        sample_key=REFERENCE_CORE_KEY,
    )
    joblib.dump(
        train_classifier(
            a.obsm["X_emb_niche"], a.obs[NICHE_KEY].astype(str).values, "MLP_niche"
        ),
        os.path.join(out_dir, "mlp_niche.joblib"),
    )
    a.obsm["X_niche_onehot"] = one_hot_annotations(a.obs[NICHE_KEY], cats)
    cc.gr.aggregate_neighbors(
        a,
        n_layers=DOMAIN_HOPS,
        use_rep="X_niche_onehot",
        out_key="X_emb_domain",
        sample_key=REFERENCE_CORE_KEY,
    )
    joblib.dump(
        train_classifier(
            a.obsm["X_emb_domain"], a.obs[DOMAIN_KEY].astype(str).values, "MLP_domain"
        ),
        os.path.join(out_dir, "mlp_domain.joblib"),
    )
    del scores, a
    print("saved", out_dir)

In [ ]:
require_new(FULL_MODEL_DIR).mkdir()
fit_transfer_bundle(str(FULL_MODEL_DIR), gene_subset=None)

## 3. Annotate the excluded reference batch

Reuse the fitted PCA and reference feature scalers. Predict niches first, then derive the domain inputs from predicted niches. Report coverage alongside accuracy over all labelled cells.

In [ ]:
def load_pca(p):
    """Load a trusted PCA artifact and its model-specific gene vocabulary."""
    z = np.load(p, allow_pickle=True)
    return (z["mean"], z["components"], list(z["genes"]))

In [ ]:
def emit_predictions(model, proba, thr):
    """Apply the probability threshold while preserving class ordering and confidence."""
    cl = np.asarray(model["classes"], object)
    j = proba.argmax(1)
    c = proba.max(1)
    lab = cl[j].astype(object)
    if thr > 0:
        lab[c < thr] = "Unknown"
    return (lab, c)

In [ ]:
def predict_reference_scale(model, emb, thr):
    """Use the saved reference standardizer for same-platform transfer."""
    return emit_predictions(model, model["pipeline"].predict_proba(emb), thr)

In [ ]:
def annotate_dataset(adata, sample_key, art_dir, predict_fn, thr=CONFIDENCE_THRESHOLD):
    """Align genes, project expression, predict niches and use predicted niches for domain inference."""
    (mean, comp, sgenes) = load_pca(os.path.join(art_dir, "pca.npz"))
    cats = json.load(open(os.path.join(art_dir, "niche_categories.json")))
    mlp_n = joblib.load(os.path.join(art_dir, "mlp_niche.joblib"))
    mlp_d = joblib.load(os.path.join(art_dir, "mlp_domain.joblib"))
    counts = sp.csr_matrix(
        adata.layers["counts"] if "counts" in adata.layers else adata.X
    )
    vi = {g: i for (i, g) in enumerate(adata.var_names)}
    if list(adata.var_names) != sgenes:
        keep = [k for (k, g) in enumerate(sgenes) if g in vi]
        cols = [vi[sgenes[k]] for k in keep]
        al = sp.lil_matrix((counts.shape[0], len(sgenes)), dtype=np.float32)
        al[:, keep] = counts[:, cols]
        counts = al.tocsr()
        if len(keep) < len(sgenes):
            print(f"⚠️ 缺 {len(sgenes) - len(keep)} 基因, 补0")
    adata.obsm["X_pca_rep"] = project_counts(log_normalize_counts(counts), mean, comp)
    adata.obs[sample_key] = (
        adata.obs[sample_key].astype("category").cat.remove_unused_categories()
    )
    sq.gr.spatial_neighbors(
        adata,
        library_key=sample_key,
        coord_type="generic",
        delaunay=True,
        spatial_key="spatial",
        percentile=99,
    )
    cc.gr.aggregate_neighbors(
        adata,
        n_layers=NICHE_HOPS,
        use_rep="X_pca_rep",
        out_key="X_emb_niche",
        sample_key=sample_key,
    )
    (ln, cn) = predict_fn(mlp_n, adata.obsm["X_emb_niche"], thr)
    adata.obs["pred_niche"] = pd.Categorical(ln)
    adata.obs["pred_niche_conf"] = cn
    adata.obsm["X_niche_onehot"] = one_hot_annotations(adata.obs["pred_niche"], cats)
    cc.gr.aggregate_neighbors(
        adata,
        n_layers=DOMAIN_HOPS,
        use_rep="X_niche_onehot",
        out_key="X_emb_domain",
        sample_key=sample_key,
    )
    (ld, cd) = predict_fn(mlp_d, adata.obsm["X_emb_domain"], thr)
    adata.obs["pred_domain"] = pd.Categorical(ld)
    adata.obs["pred_domain_conf"] = cd
    return adata

In [ ]:
def load_reference_batch(path: str | Path, batch_value: str) -> ad.AnnData:
    """Read an arbitrary held-out batch without a contiguous-row assumption."""
    (obs, xy, genes) = read_metadata(
        path, [BATCH_KEY, NICHE_KEY, DOMAIN_KEY, REFERENCE_CORE_KEY]
    )
    indices = np.flatnonzero(obs[BATCH_KEY].astype(str).to_numpy() == batch_value)
    if not len(indices):
        raise ValueError(f"Held-out batch is absent: {batch_value}")
    blocks = []
    for start, end, counts in iter_count_batches(path):
        local = indices[(indices >= start) & (indices < end)] - start
        if len(local):
            blocks.append(counts[local])
    data = ad.AnnData(sp.vstack(blocks).tocsr(), obs=obs.iloc[indices].copy())
    data.var_names = genes
    data.layers["counts"] = data.X.copy()
    data.obsm["spatial"] = xy[indices]
    return data


def evaluate_transfer(data: ad.AnnData) -> pd.DataFrame:
    """Report all-cell metrics and separately identify confidence-conditioned scores."""
    from sklearn.metrics import adjusted_rand_score, f1_score

    rows = []
    for level in ["niche", "domain"]:
        valid = data.obs[level].notna().to_numpy()
        true = data.obs.loc[valid, level].astype(str).to_numpy()
        predicted = data.obs.loc[valid, "pred_" + level].astype(str).to_numpy()
        if not len(true):
            raise ValueError(f"No held-out reference annotations for {level}.")
        confident = predicted != "Unknown"
        rows.append(
            {
                "level": level,
                "n_cells": len(true),
                "coverage": float(confident.mean()),
                "all_cell_ari": adjusted_rand_score(true, predicted),
                "all_cell_macro_f1": f1_score(
                    true,
                    predicted,
                    labels=np.unique(true),
                    average="macro",
                    zero_division=0,
                ),
                "confident_only_macro_f1": (
                    f1_score(
                        true[confident],
                        predicted[confident],
                        average="macro",
                        zero_division=0,
                    )
                    if confident.any()
                    else None
                ),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
held_out = load_reference_batch(REF_H5AD, HELD_OUT_BATCH)
held_out = annotate_dataset(
    held_out, REFERENCE_CORE_KEY, str(FULL_MODEL_DIR), predict_reference_scale
)
metrics = evaluate_transfer(held_out)

## 4. Transfer across assays using shared genes

If enabled, fit a separate reference bundle on shared genes. Align target gene order and standardize aggregated target inputs within the target dataset before classification.

In [ ]:
def standardize_target_features(M, eps=1e-08):
    """Scale aggregated classifier inputs within the target dataset for cross-platform application."""
    M = np.asarray(M, np.float32)
    mu = M.mean(0)
    sd = M.std(0)
    sd[sd < eps] = 1
    return (M - mu) / sd

In [ ]:
def predict_target_scale(model, emb, thr):
    """Bypass the reference standardizer only in the explicitly selected cross-platform mode."""
    return emit_predictions(
        model,
        model["pipeline"]
        .named_steps["m"]
        .predict_proba(standardize_target_features(emb)),
        thr,
    )

In [ ]:
if RUN_CROSS_PLATFORM:
    (_, _, reference_genes) = read_metadata(REF_H5AD, [BATCH_KEY])
    with h5py.File(COSMX_H5AD) as handle:
        target_genes = {
            g.decode() if isinstance(g, bytes) else str(g)
            for g in handle["var"]["_index"][:]
        }
    shared = [gene for gene in reference_genes if gene in target_genes]
    if len(shared) < N_COMPONENTS:
        raise ValueError("Too few shared genes for the declared PCA.")
    require_new(SHARED_MODEL_DIR).mkdir()
    fit_transfer_bundle(str(SHARED_MODEL_DIR), gene_subset=shared)
    target = annotate_dataset(
        sc.read_h5ad(COSMX_H5AD),
        TARGET_SAMPLE_KEY,
        str(SHARED_MODEL_DIR),
        predict_target_scale,
    )
    target.write_h5ad(require_new(OUT / "04_cross_platform.h5ad"))

## 5. Save predictions and evaluation tables

Save held-out annotations and metrics. The cross-assay block writes its target annotations and separate model bundle when selected.

In [ ]:
held_out.write_h5ad(require_new(OUT / "04_reference_holdout.h5ad"))
metrics.to_csv(require_new(OUT / "04_reference_metrics.csv"), index=False)
display(metrics)